# Project 2 — Local Offline AI Assistant (Small LLM)

Fully local, **no API keys**. Just run top to bottom. Emits **`RESULTS.md`** with the model comparison.

**Prereq (one-time, in a terminal):**
```bash
# install Ollama from https://ollama.com then:
ollama pull llama3.2:3b
ollama pull phi4-mini
ollama pull mistral:7b
```
The notebook auto-skips any model that isn't pulled, so it still runs with just one.

In [ ]:
%pip install -q ollama pydantic pandas

In [ ]:
import ollama
# Discover which candidate models are actually installed
CANDIDATES = ['llama3.2:3b', 'phi4-mini', 'mistral:7b']
installed = {m['model'] for m in ollama.list()['models']}
MODELS = [m for m in CANDIDATES if any(i.startswith(m) for i in installed)]
assert MODELS, 'No candidate models found. Run e.g. `ollama pull llama3.2:3b` first.'
print('Benchmarking:', MODELS)

## Phase 1 — Run + Measure
Time-to-first-token, total latency, tokens/sec — averaged over runs.

In [ ]:
import time

def benchmark(model, prompt, n_runs=3):
    ttft, total, tps = [], [], []
    for _ in range(n_runs):
        start = time.perf_counter(); first = None; n = 0
        for _chunk in ollama.generate(model=model, prompt=prompt, stream=True):
            if first is None:
                first = time.perf_counter()
            n += 1
        end = time.perf_counter()
        ttft.append(first - start); total.append(end - start); tps.append(n / (end - start))
    avg = lambda x: sum(x) / len(x)
    return {'ttft_s': round(avg(ttft), 3), 'total_s': round(avg(total), 3),
            'tokens_per_s': round(avg(tps), 1)}

print(benchmark(MODELS[0], 'Explain retrieval-augmented generation in three sentences.'))

## Phase 2 — Structure + Determinism
JSON schema + Pydantic validation + retry-once. Then temperature 0 vs 0.7 variance.

In [ ]:
from pydantic import BaseModel, ValidationError
import json

class Extraction(BaseModel):
    name: str
    sentiment: str
    score: float

def structured_generate(model, text, max_retries=1):
    prompt = (f'Extract fields as JSON matching this schema: {Extraction.model_json_schema()}.\n'
              f'Return ONLY JSON. Text: {text}')
    for attempt in range(max_retries + 1):
        resp = ollama.generate(model=model, prompt=prompt, format='json')['response']
        try:
            return Extraction.model_validate_json(resp)
        except (ValidationError, json.JSONDecodeError) as e:
            if attempt == max_retries:
                return None
            prompt += f'\nPrevious output was invalid ({e}). Return valid JSON only.'

print(structured_generate(MODELS[0], 'Alice loved the product and rated it 9 out of 10.'))

In [ ]:
# Temperature determinism study: 0.0 should be near-deterministic, 0.7 varied
PROMPT = 'Give one creative name for a coffee shop. Respond with only the name.'
temp_study = {}
for temp in (0.0, 0.7):
    outs = [ollama.generate(model=MODELS[0], prompt=PROMPT,
                            options={'temperature': temp})['response'].strip() for _ in range(5)]
    temp_study[temp] = len(set(outs))
    print(f'temp={temp}: {len(set(outs))}/5 unique -> {outs}')

## Phase 3 — Model Comparison Study
All models, same hardware: memory/params, tokens/sec, and **output quality** judged on 40 standardized prompts.

In [ ]:
TEST_PROMPTS = [
  'Summarize photosynthesis in one sentence.',
  'Write a Python function that reverses a string.',
  'What is 17 multiplied by 23?',
  'Translate "good morning" into French.',
  'List three primary colors.',
  'Explain what an API is to a 10-year-old.',
  'Write a haiku about the ocean.',
  'What is the boiling point of water at sea level in Celsius?',
  'Name the capital of Japan.',
  'Convert 100 kilometers to miles (approximately).',
  'Write a SQL query to select all rows from a table called users.',
  'What is the difference between TCP and UDP in one sentence?',
  'Give the chemical symbol for gold.',
  'Explain recursion in two sentences.',
  'What year did World War II end?',
  'Write a regular expression that matches an email address.',
  'Sort these numbers ascending: 5, 2, 9, 1.',
  'What is the largest planet in the solar system?',
  'Define the term "machine learning" in one sentence.',
  'Give a synonym for the word "happy".',
  'What is 2 to the power of 10?',
  'Write a one-line bash command to list files in the current directory.',
  'Explain the difference between a list and a tuple in Python.',
  'What is the speed of light in meters per second (approximately)?',
  'Name three programming paradigms.',
  'Write a JSON object representing a person with name and age.',
  'What does HTTP status code 404 mean?',
  'Give the past tense of the verb "run".',
  'Explain what a hash function is in one sentence.',
  'What is the freezing point of water in Fahrenheit?',
  'Write a Python list comprehension that squares numbers 1 to 5.',
  'Name the author of the play Romeo and Juliet.',
  'What is the time complexity of binary search?',
  'Convert the binary number 1010 to decimal.',
  'Give one advantage of using a database index.',
  'What is the currency of the United Kingdom?',
  'Explain the concept of a variable in programming.',
  'Write a git command to create a new branch called feature.',
  'What is the square root of 144?',
  'Define "latency" in the context of computer networks.',
]
print(len(TEST_PROMPTS), 'standardized test prompts')

In [ ]:
# Quality judge: use the largest available model to grade each answer 1-5.
JUDGE = MODELS[-1]

def judge(prompt, answer):
    q = (f'Rate the answer to a question on a 1-5 scale (5=excellent, correct, complete).\n'
         f'Question: {prompt}\nAnswer: {answer}\n'
         f'Respond with ONLY a single integer 1-5.')
    r = ollama.generate(model=JUDGE, prompt=q, options={'temperature': 0})['response']
    digits = [c for c in r if c in '12345']
    return int(digits[0]) if digits else 3

def quality_score(model):
    total = 0
    for p in TEST_PROMPTS:
        a = ollama.generate(model=model, prompt=p, options={'temperature': 0})['response']
        total += judge(p, a)
    return round(total / len(TEST_PROMPTS), 2)

print('quality scoring uses judge model:', JUDGE)

In [ ]:
import pandas as pd
rows = []
for m in MODELS:
    b = benchmark(m, 'Explain RAG in three sentences.')
    info = ollama.show(m).get('details', {})
    rows.append({'model': m,
                 'params': info.get('parameter_size', '?'),
                 'quant': info.get('quantization_level', '?'),
                 'ttft_s': b['ttft_s'], 'total_s': b['total_s'],
                 'tokens_per_s': b['tokens_per_s'],
                 'quality_1to5': quality_score(m)})
df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
# Write RESULTS.md
best_speed = df.loc[df['tokens_per_s'].idxmax(), 'model']
best_quality = df.loc[df['quality_1to5'].idxmax(), 'model']
md = ['# Project 2 — Local LLM Benchmark Report', '',
      f'Hardware: fill in (e.g. Apple M-series, RAM). Judge model: `{JUDGE}`. {len(TEST_PROMPTS)} test prompts.',
      '', '## Model Comparison', '',
      df.to_markdown(index=False), '',
      '## Determinism Study', '',
      f'- temp=0.0 -> {temp_study.get(0.0)}/5 unique outputs',
      f'- temp=0.7 -> {temp_study.get(0.7)}/5 unique outputs', '',
      '## Verdict', '',
      f'- Fastest: **{best_speed}**', f'- Highest quality: **{best_quality}**', '',
      'Structured output uses a Pydantic-validated JSON schema with retry-once-then-fail-gracefully.']
with open('RESULTS.md', 'w') as f:
    f.write('\n'.join(md))
print('Wrote RESULTS.md')